1.Bronze processing

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/gayatrijoshi663@gmail.com/regis-healthcare/1_setup/utility

In [0]:
print(bronze_schema,silver_schema,gold_schema)

In [0]:
dbutils.widgets.text("catalog","regis_healthcare","catalog")
dbutils.widgets.text("data_source","employees","data_source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")



In [0]:
# Define S3 path
bucket = "regis-healthcare"
prefix = f"source-row-data /{data_source}"
s3_path = f"s3://{bucket}/{prefix}/"

# Use dbutils to list files and find the latest
files = dbutils.fs.ls(s3_path)
latest_file = sorted(files, key=lambda x: x.modificationTime, reverse=True)[0]

base_path = latest_file.path
print("Latest file path:", base_path)


In [0]:
df = (
    spark.read.format("csv")
       .option("header",True)
       .option("inferSchema",True)
       .load(base_path)
       .withColumn("current_date",F.current_date())
       .withColumn("read_timestamp",F.current_timestamp())
       .select("*","_metadata.file_name","_metadata.file_size")
    )
print(df.count())
display(df.limit(10))

In [0]:
df.printSchema()

In [0]:
df.write\
    .format("delta")\
        .option("mergeSchema","true")\
     .option("overwriteSchema","true")\
         .option("delta.enableChangeDataFeed","true")\
             .mode("overwrite")\
                 .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

In [0]:
# bronze write to s3
df.write.format("delta")\
    .option("mergeSchema","true")\
        .option("overwriteSchema","true")\
            .mode("overwrite")\
            .partitionBy("current_date")\
            .save(f"s3://regis-healthcare/bronze-row-data/{data_source}/")

2. Silver processing

In [0]:
df_bronze =spark.sql(f"select * from {catalog}.{bronze_schema}.{data_source};")
display(df_bronze.limit(10))

In [0]:
# schema check
df_bronze.printSchema()

In [0]:
# drop duplicate
df_silver = df_bronze.dropDuplicates()


In [0]:
df_silver.columns

In [0]:

df_silver = df_silver.withColumn(
    "employee_id",
    F.trim(F.col("employee_id"))
).withColumn(
    "first_name",
    F.trim(F.col("first_name"))
).withColumn(
    "last_name",
    F.trim(F.col("last_name"))
).withColumn(
    "role",
    F.trim(F.col("role"))
).withColumn(
    "facility_id",
    F.trim(F.col("facility_id"))
).withColumn(
    "employment_type",
    F.trim(F.col("employment_type"))
).withColumn(
    "start_date",
    F.trim(F.col("start_date"))
).withColumn(
    "end_date",
    F.trim(F.col("end_date"))
).withColumn(
    "phone",
    F.trim(F.col("phone"))
).withColumn(
    "email",
    F.trim(F.col("email"))
).withColumn(
    "address",
    F.trim(F.col("address"))
).withColumn(
    "state",
    F.trim(F.col("state"))
).withColumn(
    "postcode",
    F.trim(F.col("postcode"))
).withColumn(
    "ahpra_number",
    F.trim(F.col("ahpra_number"))
).withColumn(
    "salary",
    F.trim(F.col("salary"))
).withColumn(
    "manager_id",
    F.trim(F.col("manager_id"))
).withColumn(
    "created_at",
    F.trim(F.col("created_at"))
).withColumn(
    "status",
    F.trim(F.col("status"))
)    

In [0]:
# null record count
from pyspark.sql.functions import col, count, when

null_count = df_silver.select([
    count(when(col(c).isNull(), c)).alias(c) 
    for c in df_silver.columns
])
display(null_count)

Cleaning data in table 

In [0]:
# employee_id 
check = df_silver.filter(~col("employee_id").rlike("^EMP"))
display(check)



In [0]:
#  'first_name'

# from pyspark.sql.functions import col
# df=df_silver.filter(col("first_name").isNull())

# df_silver = df_silver.fillna({
#     "first_name":"unknown"
# })

# from pyspark.sql import functions as F
# from pyspark.sql.functions import col,when

# df_invalid = df_silver.filter(col("first_name").rlike("[-_=\\[\\(<\\>\\?#*~%$&@]"))
# display(df_invalid)



In [0]:
#'last_name'


In [0]:

#  'first_name',
#  'last_name',
#  'role',
#  'facility_id',
#  'employment_type',
#  'start_date',
#  'end_date',
#  'phone',
#  'email',
#  'address',
#  'state',
#  'postcode',
#  'ahpra_number',
#  'salary',
#  'manager_id',
#  'created_at',
#  'status',